# Apache Beam — Introductory Demo

This notebook demonstrates core Beam concepts and shows the **same pipeline logic** running on two different runners:

| Runner | What it is |
|---|---|
| **DirectRunner** | Local Python process|
| **SparkRunner** | Apache Spark cluster |

> The key insight: **zero changes to business logic** when switching runners.

---
### Install (run once)
```bash
pip install apache-beam          # DirectRunner included
pip install apache-beam[spark]   # adds SparkRunner support
```

In [ ]:
! pip install apache-beam
! pip install apache-beam[interactive]
! pip install apache-beam[spark]

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
print('Apache Beam version:', beam.__version__)

---
## Helper — choose your runner

Change `RUNNER` to `"spark"` to execute every demo on Spark instead of local Python.

Everything else in the notebook stays **exactly the same**.

In [ ]:
import os

# ── CHANGE THIS to switch runners ──────────────────────────────────────────
RUNNER = "direct"   # "direct"  →  DirectRunner (local Python)
                    # "spark"   →  SparkRunner  (Spark cluster)
# ───────────────────────────────────────────────────────────────────────────

# PySpark 4.x + Java 17+ blocks access to sun.nio.ch.DirectBuffer without
# explicit module opens. Set unconditionally so every JVM child process sees it.
_jvm_opens = (
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED"
)
os.environ["JAVA_TOOL_OPTIONS"] = _jvm_opens

# Clear cached Beam job servers so the next SparkRunner run picks up the new env.
try:
    from apache_beam.runners.portability import spark_runner as _sr
    _sr.JOB_SERVER_CACHE.clear()
except Exception:
    pass

def get_options():
    """Return PipelineOptions for the chosen runner."""
    if RUNNER == "spark":
        return PipelineOptions([
            "--runner=SparkRunner",
            # "--spark_master=spark://localhost:7077",  # uncomment for real cluster
        ])
    return PipelineOptions(["--runner=DirectRunner"])

print(f'Active runner: {"DirectRunner (local Python)" if RUNNER == "direct" else "SparkRunner"}')
print(f'JAVA_TOOL_OPTIONS: {os.environ["JAVA_TOOL_OPTIONS"]}')

---
## Demo 1 — Pipeline + PCollection + Map

**Concepts covered:** `Pipeline`, `PCollection`, `beam.Create` (Source transform), `beam.Map` (PTransform)

```
beam.Create  →  PCollection(words)  →  Map(word, len)  →  PCollection(word_lengths)
```

In [ ]:
results = []

with beam.Pipeline(options=get_options()) as p:
    word_lengths = (
        p
        | "Create words"        >> beam.Create(["Apple", "Banana", "Mango", "Avocado"])
        | "Map → (word, len)"   >> beam.Map(lambda w: (w, len(w)))
      #  | "Collect"             >> beam.Map(results.append)
        | beam.Map(print)
    )

print("Output PCollection:")
for item in results:
    print(" ", item)

---
## Demo 2 — ParDo with a DoFn (User-Defined Function)

**Concepts covered:** `DoFn` (UDF), `beam.ParDo`, parallel element-wise processing

A `DoFn` is the most flexible UDF — it can emit zero, one, or many elements per input.

In [ ]:
class UpperCaseDoFn(beam.DoFn):
    """UDF: transforms each string element to uppercase."""
    def process(self, element):
        yield element.upper()


with beam.Pipeline(options=get_options()) as p:
    (
        p
        | "Input"       >> beam.Create(["hello", "world", "apache", "beam"])
        | "UpperCase"   >> beam.ParDo(UpperCaseDoFn())
        | "Collect"      >> beam.io.WriteToText("output")
    )



---
## Demo 3 — Aggregation: GroupByKey vs CombinePerKey

**Concepts covered:** `GroupByKey`, `CombinePerKey`, key-based aggregation

| Transform | Output size | Use when |
|---|---|---|
| `GroupByKey` | Same as input (groups values into lists) | You need all raw values per key |
| `CombinePerKey` | Smaller than input (one value per key) | You need a summary (sum, count, max) |

In [ ]:
sales = [
    ("India",   100), ("USA",   200), ("India", 150),
    ("USA",     300), ("India",  50), ("Germany", 400),
]

grouped_results = []
combined_results = []

with beam.Pipeline(options=get_options()) as p:
    pcoll = p | "Sales data" >> beam.Create(sales)

    # GroupByKey — keeps all raw values
    (
        pcoll
        | "GroupByKey"       >> beam.GroupByKey()
        | "Collect grouped"  >> beam.Map(lambda kv: grouped_results.append((kv[0], list(kv[1]))))
    )

    # CombinePerKey — reduces each group to a single number
    (
        pcoll
        | "SumPerKey"         >> beam.CombinePerKey(sum)
        #| "Collect combined"  >> beam.Map(combined_results.append)
        | beam.Map(print)
    )


---
## Demo 4 — Word Count 

**Concepts covered:** `FlatMap` / `ParDo`, `CombinePerKey`, composite pipeline

```
sentences  →  FlatMap(split)  →  Map(word,1)  →  CombinePerKey(sum)  →  counts
```

In [ ]:
sentences = [
    "Apache Beam is great",
    "Beam runs on Spark and Flink",
    "Beam is portable and great",
]

word_counts = []

with beam.Pipeline(options=get_options()) as p:
    (
        p
        | "Sentences"   >> beam.Create(sentences)
        | "Split"       >> beam.FlatMap(lambda s: s.lower().split())
        | "Pair with 1" >> beam.Map(lambda w: (w, 1))
        | "Count"       >> beam.CombinePerKey(sum)
        | beam.Map(print)
    )


---
## Demo 5 — Branching Pipeline (DAG)

**Concepts covered:** Pipeline as a Directed Acyclic Graph, one PCollection → multiple branches

This mirrors the slide diagram: the same PCollection feeds two independent transforms.

```
                  ┌→ Filter(starts with A) → A-names
Create(names) ───┤
                  └→ Filter(starts with B) → B-names
```

In [ ]:
names = ["Alice", "Bob", "Anna", "Charlie", "Brian", "David", "Beth"]

a_names, b_names = [], []

with beam.Pipeline(options=get_options()) as p:
    all_names = p | "Names" >> beam.Create(names)

    # Branch 1 — names starting with A
    (
        all_names
        | "Filter A"   >> beam.Filter(lambda n: n.startswith("A"))
        | "Collect A"  >> beam.io.WriteToText("output_A")
    )

    # Branch 2 — names starting with B  (same source PCollection)
    (
        all_names
        | "Filter B"   >> beam.Filter(lambda n: n.startswith("B"))
        | "Collect B"  >> beam.io.WriteToText("output_B")
    )


---
## Demo 6 — Runner Portability Side-by-Side

Run the cell below **twice** — once with `RUNNER = "direct"` and once with `RUNNER = "spark"` — to see that output is identical regardless of the runner.

> This is the core promise of Beam: **write once, run anywhere.**

In [ ]:
# Same pipeline, runner chosen by the RUNNER variable at the top of this notebook
runner_label = "DirectRunner (local Python)" if RUNNER == "direct" else "SparkRunner"
runner_label = "SparkRunner"
from apache_beam.options.pipeline_options import PipelineOptions

options = PipelineOptions([
    '--runner=SparkRunner',
    '--spark_master_url=local[*]'
])
with beam.Pipeline(options=options) as p:
    (
        p
        | beam.Create(range(1, 6))
        | beam.Map(lambda x: ("odd" if x % 2 else "even", x))
        | beam.CombinePerKey(sum)
        | beam.Map(print)
    )


In [ ]:
import apache_beam as beam
import grpc

print("Beam:", beam.__version__)
print("gRPC:", grpc.__version__)

---
## Summary

| Concept | Demo | Key takeaway |
|---|---|---|
| Pipeline, PCollection, Map | Demo 1 | Data flows as PCollections between transforms |
| DoFn / UDF | Demo 2 | Custom logic lives in `DoFn.process()` |
| GroupByKey vs CombinePerKey | Demo 3 | Combine reduces data; GroupByKey does not |
| Word Count | Demo 4 | Classic batch pipeline built from primitives |
| Branching DAG | Demo 5 | One PCollection can feed multiple transforms |
| Runner portability | Demo 6 | Change one variable to switch execution engines |